# Hướng Dẫn Giải Thích Chi Tiết: `src/web/` và `src/web_runner/`

Notebook này phân tích cấu trúc, công dụng và cách hoạt động của nền tảng Web dự đoán giá cổ phiếu. Nền tảng được xây dựng dựa trên sự kết hợp giữa **FastAPI (Backend)**, **SQLite/SQLAlchemy (Database)** và **Chart.js (Frontend)**.

---

## 🌐 1. Cấu Trúc Thành Phần Web

Hệ thống web được chia nhỏ thành các cấu phần chuyên biệt nằm trong thư mục `src/web/`:
- **`src/web/backend/db.py`**: Định nghĩa cấu trúc cơ sở dữ liệu SQLite thông qua ORM SQLAlchemy. Quản lý 4 bảng dữ liệu chính.
- **`src/web/backend/api.py`**: Khai báo các API endpoints RESTful bằng FastAPI. Chứa logic kích hoạt dự đoán trực tuyến (`Real-time prediction trigger`).
- **`src/web/frontend/`**: Giao diện người dùng thuần chất lượng cao (HTML, Vanilla CSS, JS với biểu đồ Chart.js).
- **`src/web_runner/run_web.py`**: Điểm khởi động (Entry point) khởi tạo DB, đồng bộ dữ liệu lịch sử và khởi chạy server Uvicorn.

In [1]:
import sys
import os

# Thêm thư mục gốc vào đường dẫn hệ thống để import src
ROOT_DIR = os.path.abspath('..')
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

print(f"Thư mục gốc: {ROOT_DIR}")

Thư mục gốc: C:\Users\ACER\Documents\Stock-Opening-Price-Prediction


## 🗄️ 2. Thiết Kế Cơ Sở Dữ Liệu (`db.py`)

Cơ sở dữ liệu SQLite nằm tại `data/stock_predictions.db` quản lý 4 bảng liên kết khóa ngoại:
1. **`stocks`**: Lưu trữ danh sách cổ phiếu cần theo dõi (Watchlist) bao gồm `ticker`, `name` và `currency`.
2. **`stock_prices`**: Lưu trữ lịch sử giá đóng/mở/cao/thấp hàng ngày phục vụ vẽ biểu đồ.
3. **`news_sentiments`**: Lưu trữ tiêu đề tin tức và điểm số cảm xúc của tin tức tương ứng.
4. **`prediction_records`**: Lưu trữ các dự đoán mở cửa của XGBoost và Transformer cùng dải an toàn và mức độ rủi ro tương ứng.

In [2]:
from src.web.backend.db import SessionLocal, Stock, StockPrice, PredictionRecord

db = SessionLocal()
try:
    stocks = db.query(Stock).all()
    print("Watchlist hiện tại trong SQLite Database:")
    for s in stocks:
        prices_count = db.query(StockPrice).filter(StockPrice.stock_id == s.id).count()
        preds_count = db.query(PredictionRecord).filter(PredictionRecord.stock_id == s.id).count()
        print(f" - {s.ticker} ({s.name}) | Giá lịch sử: {prices_count} phiên | Số bản ghi dự đoán: {preds_count}")
finally:
    db.close()

Watchlist hiện tại trong SQLite Database:
 - VNM.VN (Vinamilk) | Giá lịch sử: 131 phiên | Số bản ghi dự đoán: 17
 - GOOGL (Alphabet Inc.) | Giá lịch sử: 124 phiên | Số bản ghi dự đoán: 10
 - META (Meta Platforms) | Giá lịch sử: 124 phiên | Số bản ghi dự đoán: 8


## 🔌 3. REST API Endpoints (`api.py`)

FastAPI cung cấp các endpoints phục vụ giao diện người dùng:
- **`GET /api/health`**: Kiểm tra trạng thái hoạt động của server.
- **`GET /api/stocks`**: Lấy danh sách Watchlist.
- **`GET /api/prices/{ticker}`**: Lấy dữ liệu giá lịch sử để vẽ biểu đồ trên giao diện.
- **`GET /api/predictions/{ticker}`**: Lấy lịch sử các dự báo của mô hình AI.
- **`GET /api/news/{ticker}`**: Lấy tin tức và chỉ số cảm xúc tin tức gần nhất.
- **`POST /api/predict/trigger/{ticker}`**: API Kích hoạt Dự đoán trực tuyến.

## ⚡ 4. Quy Trình Kích Hoạt Dự Đoán Trực Tuyến (`Real-time Prediction Trigger`)

Khi người dùng click vào nút **"Kích hoạt Dự báo Online"** trên giao diện, endpoint `/api/predict/trigger/{ticker}` thực hiện các bước sau:
1. **Tải mô hình động:** Đọc các tệp `.pkl` và `.keras` của mã cổ phiếu tương ứng từ thư mục `models/`.
2. **Tải giá trực tuyến:** Sử dụng `yfinance` tải 150 ngày giá mới nhất.
3. **Tải tin tức:** Gọi engine phân tích cảm xúc tin tức (`VADER` hoặc `FinBERT`) để chấm điểm tin tức ngày hôm nay.
4. **Tính toán đặc trưng:** Tạo tập 24 đặc trưng đồng bộ hoàn toàn với quá trình huấn luyện ngoại tuyến.
5. **Suy diễn (Inference):** Đưa qua XGBoost và Transformer để lấy dự đoán mở cửa của phiên kế tiếp.
6. **Đánh giá rủi ro:** Tính toán độ dao động ATR để đưa ra mức độ rủi ro (`Thấp`/`Trung bình`/`Cao`) và khoảng an toàn biến động.
7. **Lưu trữ:** Ghi lại kết quả dự án mới nhất vào DB để frontend tự động cập nhật biểu đồ và bảng kết quả.

## 🚀 5. Điểm Khởi Chạy `run_web.py`

Khi chạy `python src/web_runner/run_web.py`:
1. Hệ thống sẽ gọi hàm khởi tạo DB `init_db()` để tạo các bảng nếu chưa tồn tại.
2. Tải giá lịch sử 6 tháng gần nhất từ yfinance và lưu vào DB để đảm bảo biểu đồ hoạt động ngay lập tức.
3. Đọc tệp log cũ `reports/figures/predictions_history.txt` để khôi phục và chuyển đổi toàn bộ lịch sử dự đoán cũ vào cơ sở dữ liệu SQLite.
4. Chạy Uvicorn Web Server lắng nghe tại địa chỉ `http://127.0.0.1:8000/` và tự động gắn kết (mount) thư mục static `src/web/frontend/` vào URL gốc.